# 02 — Lake area over time

**In plain words:** notebook 01 told us *which* lakes exist. This notebook measures **how big each lake was over
time**, because a lake that keeps growing is a warning sign.

| Output | What it is |
| --- | --- |
| `lake_area_yearly.csv` | for **all 552 lakes**, the largest water area in each melt season (May–October), **2000–2025**, from Landsat |
| `lake_area_weekly.csv` | for the **3 Tier 2 lakes**, the water area **every week, 2015–2025**, from Sentinel-1 radar (sees through cloud) |
| `s1_acquisitions.csv` | every single radar measurement behind the weekly table |
| `event_response.csv` | the check: did each lake's weekly area **drop** when a recorded outburst happened? |
| `fig_02_tier2_weekly.png`, `fig_02_tier2_yearly.png`, `run_info.json` | figures, and a record of versions, parameters and counts |

**Steps:** 1 load the lakes from notebook 01 → 2 yearly area from Landsat → 3 weekly area from radar → 4 checks → 5 save → 6 figures.

**How to run:** open from the GitHub link, Runtime → Run all, approve the pop-ups. Send back the cells marked
**📋 REPORT** and the two figures.

## 0. Setup
Same as notebook 01: use the tested code in `src/karaglof/` from the GitHub repository. The lake list is read from
the repository too (`data/labels/`), so this notebook uses exactly the committed v0.2 lakes.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/inamjadoon/KaraGlof.git"
REPO_DIR = "/content/KaraGlof"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
import json
import time
from datetime import UTC, datetime
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import drive

from karaglof import area as ka

ee.Authenticate()
ee.Initialize(project="glof-497313")

drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/KaraGlof/02_lake_area")
OUT.mkdir(parents=True, exist_ok=True)
LABELS = Path(REPO_DIR) / "data" / "labels"
EXPORT_FOLDER = "KaraGlof_exports"
print("outputs ->", OUT)

## Parameters
Every choice in one place, so each one can be named and defended.
* **Yearly (Landsat only, all years).** Using the *same* satellite family every year means a lake cannot appear to
  "grow" in 2017 just because a sharper satellite (Sentinel-2) started then. Sentinel-2 is therefore not used for the
  yearly record (and never before 2017, a project rule).
* **Season May–October:** the melt season. It includes the spring filling of ice-dammed lakes such as Shisper, which
  burst in May–June.
* **Water test:** NDWI = (green − near-infrared) / (green + near-infrared) above `NDWI_MIN`, seen in at least
  `MIN_DETECTIONS` clear looks. One look can be fooled by a shadow; two independent looks rarely are.
* **No silent filling:** a lake-year with fewer than `MIN_CLEAR_OBS` clear looks gets an **empty** area.
* **Radar:** calm water reflects the radar beam away, so it looks dark (low VV backscatter). Pixels darker than
  `S1_VV_WATER_DB` on flat ground (slope < `S1_MAX_SLOPE_DEG`, because steep slopes cause radar shadow) count as water.

In [ ]:
YEARS = list(range(2000, 2026))
SEASON = ("05-01", "11-01")              # May 1 to Oct 31 (end date is exclusive)
LAKE_BUFFER_M = 60                       # look 2 Landsat pixels around each outline, so growth is captured
TIER2_BUFFER_M = 300                     # Tier 2 lakes can grow far beyond their mapped outline
NDWI_MIN = 0.1                           # water threshold for Landsat surface reflectance
MIN_DETECTIONS = 2                       # a pixel counts as water if seen as water in >= 2 clear looks
MIN_CLEAR_OBS = 3                        # fewer clear looks in a season -> area left empty (not filled)

S1_START, S1_END = "2015-01-01", "2026-01-01"
S1_VV_WATER_DB = -18.0                   # radar: darker than -18 dB = calm water
S1_SPECKLE_M = 30                        # smooth radar "speckle" noise over 30 m before thresholding
S1_MAX_SLOPE_DEG = 15                    # only flat ground (lakes are flat; slopes cause radar shadow)
S1_MIN_VALID_FRAC = 0.9                  # use an image only if it covers >= 90% of the lake area
EVENT_WINDOW_WEEKS = 4                   # check: mean area 4 weeks before vs 4 weeks after each outburst
UTM = "EPSG:32643"

## Helper: bring Earth Engine results into Python
The same cached-export helper as notebook 01: big results are exported to Drive (they keep running if Colab
disconnects; watch them at <https://code.earthengine.google.com/tasks>), and a rerun reads the saved file instead of
recomputing it.

In [ ]:
EXPORTS = Path("/content/drive/MyDrive") / EXPORT_FOLDER

def export(fc, name, fmt="CSV"):
    path = EXPORTS / f"{name}.{fmt.lower()}"
    if path.exists():
        print(f"{name}: cached file found, nothing to recompute")
        return path, None
    desc = f"karaglof_{name}"
    for t in ee.batch.Task.list()[:30]:
        st = t.status()
        if st.get("description") == desc and st.get("state") in ("READY", "RUNNING"):
            print(f"{name}: the same export is already running, waiting for it")
            return path, t
    task = ee.batch.Export.table.toDrive(collection=fc, description=desc, folder=EXPORT_FOLDER,
                                         fileNamePrefix=name, fileFormat=fmt)
    task.start()
    print(f"{name}: export started (watch it at https://code.earthengine.google.com/tasks)")
    return path, task

def read_export(path, task):
    t0 = time.time()
    while task is not None and task.active():
        print(f"\r{path.stem}: {task.status()['state']} for {(time.time() - t0) / 60:.0f} min ", end="")
        time.sleep(30)
    if task is not None and task.status()["state"] != "COMPLETED":
        raise RuntimeError(task.status())
    for _ in range(60):
        if path.exists():
            break
        time.sleep(10)
    print(f"\n{path.stem}: ready")
    return pd.read_csv(path)

def to_ee(gdf):
    # lake outlines (lon/lat, rounded to ~0.1 m) as an Earth Engine collection with only the lake_id
    small = gdf[["lake_id", "geometry"]].copy()
    small["geometry"] = small.geometry.set_precision(1e-6)
    gj = json.loads(small.to_json())
    return ee.FeatureCollection([ee.Feature(f) for f in gj["features"]])

## 1. Load the lakes from notebook 01
Reads the committed v0.2 tables. Each outline is widened by `LAKE_BUFFER_M` (Tier 2 lakes by `TIER2_BUFFER_M`), so
water that spreads beyond the mapped outline in a big year is still counted. Outlines are simplified to 5 m (much
finer than a 30 m Landsat pixel) to keep the upload to Earth Engine small.

In [ ]:
lakes = gpd.read_file(LABELS / "lakes_polygons.geojson")
events = pd.read_csv(LABELS / "events.csv")
tier2 = lakes[lakes.tier2].copy()
TIER2_IDS = dict(zip(tier2.name, tier2.lake_id, strict=True))

utm = lakes.to_crs(UTM)
buffer_m = pd.Series(LAKE_BUFFER_M, index=lakes.index).where(~lakes.tier2, TIER2_BUFFER_M)
zones = gpd.GeoDataFrame(lakes[["lake_id", "tier2"]].copy(),
                         geometry=[g.buffer(b).simplify(5) for g, b in zip(utm.geometry, buffer_m, strict=True)],
                         crs=UTM).to_crs("EPSG:4326")
lakes_fc = to_ee(zones)
t2_fc = to_ee(zones[zones.tier2])
box = ee.Geometry.Rectangle(list(zones.total_bounds))
t2_box = ee.Geometry.Rectangle(list(zones[zones.tier2].total_bounds))
print(f"{len(lakes)} lakes ({len(tier2)} Tier 2: {TIER2_IDS}); {len(events)} events")   # 📋 REPORT

## 2. Yearly lake area from Landsat (all lakes, 2000–2025)
For each year, every Landsat 5, 7, 8 and 9 image of the melt season is used. For each image:
* **clear** = not cloud, not cloud shadow (the image's own quality band, `QA_PIXEL`);
* **water** = clear and NDWI > `NDWI_MIN`.

Over the season, a pixel is water if it was water in at least `MIN_DETECTIONS` looks. The lake's yearly area is the
total area of those pixels inside its zone. The number of clear looks over the lake is recorded too, so thin years are
visible and left empty rather than guessed. Landsat 7 has striped gaps after 2003; combining all looks of a season
fills them naturally.

In [ ]:
LANDSAT = [("LANDSAT/LT05/C02/T1_L2", "SR_B2", "SR_B4"),   # Landsat 5: green, near-infrared
           ("LANDSAT/LE07/C02/T1_L2", "SR_B2", "SR_B4"),   # Landsat 7
           ("LANDSAT/LC08/C02/T1_L2", "SR_B3", "SR_B5"),   # Landsat 8
           ("LANDSAT/LC09/C02/T1_L2", "SR_B3", "SR_B5")]   # Landsat 9

def landsat_looks(year):
    start, end = f"{year}-{SEASON[0]}", f"{year}-{SEASON[1]}"
    merged = None
    for cid, green, nir in LANDSAT:
        def look(img, green=green, nir=nir):
            clear = img.select("QA_PIXEL").bitwiseAnd(0b11010).eq(0)    # bits: dilated cloud, cloud, shadow
            sr = img.select([green, nir]).multiply(0.0000275).add(-0.2)  # Collection 2 scale factors
            water = sr.normalizedDifference([green, nir]).gt(NDWI_MIN).And(clear)
            return water.rename("water").addBands(clear.rename("clear")).toUint8()
        col = ee.ImageCollection(cid).filterBounds(box).filterDate(start, end).map(look)
        merged = col if merged is None else merged.merge(col)
    return merged

pixel_km2 = ee.Image.pixelArea().divide(1e6)
images_per_year, yearly_fcs = {}, []
for year in YEARS:
    looks = landsat_looks(year)
    images_per_year[year] = looks.size().getInfo()
    if images_per_year[year] == 0:
        continue
    counts = looks.sum()
    water_km2 = counts.select("water").gte(MIN_DETECTIONS).multiply(pixel_km2).rename("water_km2")
    fc = water_km2.reduceRegions(lakes_fc, ee.Reducer.sum(), 30)
    fc = counts.select("clear").reduceRegions(fc, ee.Reducer.median(), 30)
    yearly_fcs.append(fc.map(lambda f, year=year: ee.Feature(None, {
        "lake_id": f.get("lake_id"), "year": year, "water_km2": f.get("sum"), "clear_obs": f.get("median")})))
print("Landsat images per melt season:", images_per_year)   # 📋 REPORT

TAG = f"ndwi{NDWI_MIN}_det{MIN_DETECTIONS}_buf{LAKE_BUFFER_M}-{TIER2_BUFFER_M}"
raw_yearly = read_export(*export(ee.FeatureCollection(yearly_fcs).flatten(), f"lake_area_yearly_raw_{TAG}"))
yearly = ka.yearly_table(raw_yearly, lakes.lake_id, YEARS, MIN_CLEAR_OBS)
cov = yearly.groupby("year").area_km2.apply(lambda s: s.notna().mean()).round(2)
print(f"lake-years with an area: {yearly.area_km2.notna().mean():.1%}")   # 📋 REPORT
print("share of lakes with an area, per year:", cov.to_dict())

## 3. Weekly lake area from Sentinel-1 radar (Tier 2 lakes, 2015–2025)
Optical satellites are blind under cloud, and outbursts happen in the cloudy melt season. Radar sees through cloud.
For every Sentinel-1 image (one every ~6–12 days):
1. smooth the radar speckle over `S1_SPECKLE_M`;
2. **water** = darker than `S1_VV_WATER_DB`, on flat ground (slope from the Copernicus 30 m elevation model);
3. record the water area and how much of the lake zone the image covered (`valid_frac`).

`karaglof.area.weekly_series` then keeps images that cover at least `S1_MIN_VALID_FRAC` of the lake and takes the
**median per week** (Monday to Sunday), which resists a single noisy image. Weeks without a usable image stay empty.
**Known limitation:** wet snow is also dark on radar, so early-spring weeks can overstate water. The checks in
step 4 show how much this matters.

In [ ]:
dem_col = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM")
dem = dem_col.mosaic().setDefaultProjection(dem_col.first().projection())
flat = ee.Terrain.slope(dem).lt(S1_MAX_SLOPE_DEG)
flat_km2 = {f["properties"]["lake_id"]: f["properties"]["sum"]
            for f in flat.multiply(pixel_km2).reduceRegions(t2_fc, ee.Reducer.sum(), 10).getInfo()["features"]}
print("flat area in each Tier 2 zone (km²):", {k: round(v, 3) for k, v in flat_km2.items()})

s1 = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(t2_box).filterDate(S1_START, S1_END)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV")).select("VV"))

def s1_look(img):
    covered = img.mask().gt(0).And(flat)
    water = img.focalMedian(S1_SPECKLE_M, "circle", "meters").lt(S1_VV_WATER_DB).And(covered)
    bands = ee.Image.cat(water.multiply(pixel_km2).rename("water_km2"),
                         covered.multiply(pixel_km2).rename("valid_km2"))
    return bands.reduceRegions(t2_fc, ee.Reducer.sum(), 10).map(lambda f: f.set({
        "date": img.date().format("YYYY-MM-dd"), "orbit": img.get("orbitProperties_pass")}))

S1_TAG = f"vv{S1_VV_WATER_DB:g}_sp{S1_SPECKLE_M}_sl{S1_MAX_SLOPE_DEG}_buf{TIER2_BUFFER_M}"
acq = read_export(*export(s1.map(s1_look).flatten().select(["lake_id", "date", "orbit", "water_km2", "valid_km2"]),
                          f"s1_tier2_{S1_TAG}"))
acq["valid_frac"] = acq.valid_km2 / acq.lake_id.map(flat_km2)
weekly = ka.weekly_series(acq, TIER2_IDS.values(), S1_START, "2025-12-31", S1_MIN_VALID_FRAC)
print(f"{len(acq)} radar measurements; {(acq.valid_frac >= S1_MIN_VALID_FRAC).mean():.0%} cover the lake well")   # 📋 REPORT
print("weeks with data, per lake:", weekly.groupby("lake_id").area_km2_s1.apply(lambda s: int(s.notna().sum())).to_dict())

## 4. Checks 📋 REPORT
1. **Does the radar see the outbursts?** For every outburst since 2015 with an exact day, compare the mean weekly
   area in the 4 weeks before with the 4 weeks after. A real drainage should show a clear **drop** (negative
   `change_pct`). This checks the *data*; no threshold is tuned on it.
2. **Does the radar agree with independent maps?** The GLOF database mapped each lake on Planet images just before
   outbursts (carried over from notebook 01). The closest radar measurement within 6 days is compared with it.
3. **No silent filling:** every empty lake-year has fewer than `MIN_CLEAR_OBS` clear looks, and nothing is marked
   filled.

In [ ]:
ev = events[(events.date_certainty == "day") & (events.date >= S1_START) & events.lake_id.isin(TIER2_IDS.values())]
response = ka.event_response(weekly, ev, EVENT_WINDOW_WEEKS)
response["lake"] = response.lake_id.map({v: k for k, v in TIER2_IDS.items()})
print(response[["lake", "date", "weeks_with_data_before", "weeks_with_data_after",
                "before_km2", "after_km2", "change_pct"]].to_string(index=False))
drops = response.dropna(subset=["change_pct"])
print(f"\noutbursts with a drop of at least 30%: {int((drops.change_pct <= -30).sum())} of {len(drops)} measurable")

ref = pd.DataFrame(json.loads((LABELS / "run_info.json").read_text())["tier2_outline_validation"])
ref = ref[ref.db_area_km2.notna() & (ref.db_image != "nan")].copy()
acq_ok = acq[acq.valid_frac >= S1_MIN_VALID_FRAC].copy()
acq_ok["date"] = pd.to_datetime(acq_ok.date)
rows = []
for r in ref.itertuples():
    a = acq_ok[acq_ok.lake_id == TIER2_IDS[r.lake]]
    gap = (a.date - pd.Timestamp(r.db_image)).abs()
    near = a[gap <= pd.Timedelta(days=6)]
    best = near.loc[gap[near.index].idxmin()] if len(near) else None
    rows.append({"lake": r.lake, "planet_image": r.db_image, "planet_km2": r.db_area_km2,
                 "radar_date": best.date.date() if best is not None else None,
                 "radar_km2": round(best.water_km2, 3) if best is not None else None})
print("\nRadar vs independent Planet-mapped area before outbursts:")
print(pd.DataFrame(rows).to_string(index=False))

empty = yearly[yearly.area_km2.isna()]
assert (empty.n_clear_obs < MIN_CLEAR_OBS).all() and not yearly.filled.any(), "a gap was filled silently"
print(f"\nNo silent filling: {len(empty)} empty lake-years, all with < {MIN_CLEAR_OBS} clear looks. PASS")

## 5. Save outputs to Drive

In [ ]:
yearly.to_csv(OUT / "lake_area_yearly.csv", index=False)
weekly.assign(week_start=weekly.week_start.dt.strftime("%Y-%m-%d")).to_csv(OUT / "lake_area_weekly.csv", index=False)
acq.to_csv(OUT / "s1_acquisitions.csv", index=False)
response.to_csv(OUT / "event_response.csv", index=False)
run_info = {
    "notebook": "02_lake_area", "run_utc": datetime.now(UTC).isoformat(timespec="seconds"),
    "sources": {"yearly": "Landsat 5/7/8/9 Collection 2 Level 2 (LANDSAT/L*0*/C02/T1_L2)",
                "weekly": "Sentinel-1 GRD IW VV (COPERNICUS/S1_GRD)", "slope": "COPERNICUS/DEM/GLO30",
                "lakes": "data/labels (notebook 01, v0.2)"},
    "parameters": {k: v for k, v in globals().items() if k.isupper() and k not in ("OUT", "REPO_DIR", "LABELS")
                   and isinstance(v, int | float | str | list | tuple | dict)},
    "counts": {"landsat_images_per_year": images_per_year, "lake_years": len(yearly),
               "lake_years_with_area": int(yearly.area_km2.notna().sum()),
               "radar_measurements": len(acq), "weeks_with_radar_area": int(weekly.area_km2_s1.notna().sum())},
    "event_response": response.to_dict(orient="records"),
    "versions": {"earthengine-api": ee.__version__, "geemap": geemap.__version__, "pandas": pd.__version__},
}
(OUT / "run_info.json").write_text(json.dumps(run_info, indent=2, default=str))
print(sorted(p.name for p in OUT.iterdir()))

## 6. Figures 📋 REPORT
**Weekly figure:** each Tier 2 lake's radar water area 2015–2025. Red lines are recorded outbursts, orange dots the
independent Planet-mapped areas. Look for: the lake **filling up before** each red line and **dropping right after**.
**Yearly figure:** the same lakes' largest Landsat area each melt season since 2000.

In [ ]:
names = {v: k for k, v in TIER2_IDS.items()}
fig, axes = plt.subplots(len(TIER2_IDS), 1, figsize=(13, 3.2 * len(TIER2_IDS)), sharex=True)
for ax, (name, lid) in zip(axes, TIER2_IDS.items(), strict=True):
    w = weekly[weekly.lake_id == lid]
    ax.plot(w.week_start, w.area_km2_s1, lw=1, color="#1f77b4", label="radar weekly area")
    for d in events[events.lake_id == lid].date:
        if d >= S1_START:
            ax.axvline(pd.Timestamp(d), color="red", lw=1, alpha=0.7)
    r = ref[ref.lake == name]
    ax.scatter(pd.to_datetime(r.db_image), r.db_area_km2, color="orange", zorder=3, label="Planet-mapped (database)")
    ax.set_title(f"{name} ({lid}): weekly water area, red = recorded outburst")
    ax.set_ylabel("km²")
    ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "fig_02_tier2_weekly.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(13, 4))
for name, lid in TIER2_IDS.items():
    y = yearly[yearly.lake_id == lid]
    ax.plot(y.year, y.area_km2, marker="o", label=name)
ax.set_title("Tier 2 lakes: largest water area each melt season (Landsat)")
ax.set_xlabel("year")
ax.set_ylabel("km²")
ax.legend()
fig.savefig(OUT / "fig_02_tier2_yearly.png", dpi=150)
plt.show()

## What to send back
1. The output of every cell marked **📋 REPORT** (steps 1–4).
2. Both figures.

As before: if a cell shows an error, stop there and send it, and don't change thresholds yourself. The weekly
radar threshold in particular must be settled **before** notebook 05, and never adjusted after looking at the
Shisper 2022 replay.